# 🏥 ICD-Insight — Notebook A: QLoRA Training
### BioClinical ModernBERT + QLoRA Fine-tuning on Public Hugging Face Dataset

> **M.Tech Final Year Project** | ICD-10 Multi-Label Classification  
> Run on **Google Colab Free Tier (T4 GPU)**

---

**What this notebook does:**
1. Installs all dependencies
2. Mounts Google Drive for checkpoint saving
3. Downloads and preprocesses `birgermoell/icd10-clinical-notes` (100% public, no DUA needed)
4. Fine-tunes BioClinical ModernBERT with **QLoRA** (4-bit quantized base + LoRA adapters)
5. Evaluates on held-out test set
6. Saves LoRA adapter weights (~20–50 MB) to Drive for GitHub upload

**Prerequisites:**
- ✅ Clone this repo to colab so `src/preprocess.py` is available
- ✅ Runtime set to **GPU → T4** (Runtime → Change runtime type → T4 GPU)

## A.0 — Hardware Check

In [ ]:
import subprocess, torch

print("=" * 60)
print("GPU INFO")
print("=" * 60)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"  GPU Name  : {gpu.name}")
    print(f"  VRAM      : {gpu.total_memory / 1e9:.1f} GB")
    print(f"  CUDA ver  : {torch.version.cuda}")
else:
    print("  ⚠ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

result = subprocess.run(["df", "-h", "/"], capture_output=True, text=True)
print("\nDISK SPACE")
print(result.stdout)

import psutil
ram = psutil.virtual_memory()
print(f"RAM       : {ram.total / 1e9:.1f} GB total, {ram.available / 1e9:.1f} GB available")

## A.1 — Install Dependencies

> ⏱ Takes ~3–5 minutes on first run. Restart runtime if prompted.

In [ ]:
%%capture
!pip install -q \
    transformers==4.41.0 \
    peft==0.11.0 \
    bitsandbytes==0.43.1 \
    accelerate==0.30.0 \
    datasets==2.19.0 \
    scikit-learn==1.4.2 \
    evaluate==0.4.2 \
    pymupdf==1.24.3 \
    reportlab==4.2.0 \
    psutil \
    wandb

print("✅ Dependencies installed")

In [ ]:
import torch
import transformers
import peft
import bitsandbytes as bnb

print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"bitsandbytes : {bnb.__version__}")
print(f"CUDA avail   : {torch.cuda.is_available()}")

## A.2 — Mount Google Drive

Checkpoints are saved here mid-training so they survive session timeouts.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/icd-insight/checkpoints"
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f"✅ Drive mounted — checkpoints will save to: {DRIVE_CHECKPOINT_DIR}")

## A.3 — Dataset Download & Preprocessing

We will use the public `birgermoell/icd10-clinical-notes` dataset from Hugging Face.  
To run the preprocessing script, we first need to clone our GitHub repo.

In [ ]:
# ─── CONFIGURATION — Edit these values ───────────────────────────────────────

BASE_MODEL_ID  = "NLP4Science/BioClinical-ModernBERT-base"
TOP_K_CODES    = 50          # number of most-frequent ICD-10 codes to target
MAX_NOTES      = 60000       # total notes to use (train+val+test)
MAX_LENGTH     = 2048        # token length — safe for T4 16 GB with QLoRA
BATCH_SIZE     = 4
GRAD_ACCUM     = 8           # effective batch = 4 × 8 = 32
LEARNING_RATE  = 2e-4
NUM_EPOCHS     = 5
LORA_R         = 16
LORA_ALPHA     = 32
THRESHOLD      = 0.5         # sigmoid threshold for positive label
SEED           = 42

LOCAL_DATA_DIR = "/content/data/processed"
LOCAL_CKPT_DIR = "/content/checkpoints"
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)

print("Configuration:")
print(f"  Base model  : {BASE_MODEL_ID}")
print(f"  Max length  : {MAX_LENGTH} tokens")
print(f"  Batch size  : {BATCH_SIZE} (× {GRAD_ACCUM} grad accum = {BATCH_SIZE*GRAD_ACCUM} effective)")
print(f"  LoRA rank   : r={LORA_R}, alpha={LORA_ALPHA}")
print(f"  Epochs      : {NUM_EPOCHS}")

In [ ]:
# Clone the repo to get src/preprocess.py
import os, subprocess
REPO_DIR = "/content/icd-insight"
GITHUB_REPO = "https://github.com/YOUR_USERNAME/icd-insight.git" # Update if needed

if not os.path.exists(REPO_DIR):
    print(f"▶ Cloning {GITHUB_REPO} …")
    subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR], check=True)
else:
    print("▶ Repo already cloned.")

print("\n▶ Running preprocessing pipeline …")
!python {REPO_DIR}/src/preprocess.py \
    --output_dir {LOCAL_DATA_DIR} \
    --top_k_codes {TOP_K_CODES} \
    --max_notes {MAX_NOTES}


## A.4 — Tokenization & Dataset Class

In [ ]:
import torch
import json
import pickle
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

class ICD10Dataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, mlb, max_length=2048):
        self.tokenizer = tokenizer
        self.mlb = mlb
        self.max_length = max_length
        self.records = []
        with open(jsonl_path) as f:
            for line in f:
                self.records.append(json.loads(line))

    def __len__(self): return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        enc = self.tokenizer(
            rec["text"], max_length=self.max_length,
            truncation=True, padding="max_length", return_tensors="pt"
        )
        label_vec = self.mlb.transform([rec["labels"]])[0]
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(label_vec, dtype=torch.float),
        }

print("▶ Loading tokenizer …")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
print(f"  Vocab size : {tokenizer.vocab_size:,}")
print(f"  Max length : {MAX_LENGTH}")

# Load MultiLabelBinarizer
MLB_PATH = f"{LOCAL_DATA_DIR}/label_binarizer.pkl"
with open(MLB_PATH, "rb") as f:
    MLB = pickle.load(f)

train_ds = ICD10Dataset(f"{LOCAL_DATA_DIR}/train.jsonl", tokenizer, MLB, MAX_LENGTH)
val_ds   = ICD10Dataset(f"{LOCAL_DATA_DIR}/val.jsonl",   tokenizer, MLB, MAX_LENGTH)
test_ds  = ICD10Dataset(f"{LOCAL_DATA_DIR}/test.jsonl",  tokenizer, MLB, MAX_LENGTH)

print(f"  Train examples : {len(train_ds):,}")
print(f"  Val examples   : {len(val_ds):,}")
print(f"  Test examples  : {len(test_ds):,}")

# Quick check
sample = train_ds[0]
print(f"  Sample input_ids shape : {sample['input_ids'].shape}")
print(f"  Sample labels shape    : {sample['labels'].shape}")

## A.5 — Load Model in 4-bit (QLoRA)

Base model loaded in NF4 quantization (~2 GB VRAM).  
LoRA adapters attached (~4–8 M trainable parameters out of 150 M total).

In [ ]:
from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

NUM_LABELS = len(MLB.classes_)
print(f"▶ Loading {BASE_MODEL_ID} in 4-bit NF4 ({NUM_LABELS} labels) …")

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load base model
base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_ID,
    num_labels=NUM_LABELS,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    problem_type="multi_label_classification",
)

# Prepare for k-bit training
base_model = prepare_model_for_kbit_training(base_model)

# LoRA configuration
lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["self_attn.q_proj", "self_attn.k_proj",
                    "self_attn.v_proj", "self_attn.o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved() / 1e9
    print(f"\n  VRAM allocated : {allocated:.2f} GB")
    print(f"  VRAM reserved  : {reserved:.2f} GB")

## A.6 — Training Configuration

In [ ]:
import torch.nn as nn

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        prob = torch.sigmoid(logits)
        p_t  = prob * targets + (1 - prob) * (1 - targets)
        return ((1 - p_t) ** self.gamma * bce).mean()

focal_loss_fn = FocalLoss(gamma=2.0)
print("✅ Focal loss ready (gamma=2.0)")

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate, numpy as np
from sklearn.metrics import f1_score, roc_auc_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs >= THRESHOLD).astype(int)

    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)

    try:
        auc = roc_auc_score(labels, probs, average="micro")
    except Exception:
        auc = 0.0

    return {"micro_f1": micro_f1, "macro_f1": macro_f1, "auc_roc": auc}


class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss    = focal_loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


training_args = TrainingArguments(
    output_dir=LOCAL_CKPT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.10,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print("✅ Trainer configured")
steps_per_epoch = len(train_ds) // (BATCH_SIZE * GRAD_ACCUM)
print(f"   Effective steps/epoch : {steps_per_epoch:,}")
print(f"   Total training steps  : {steps_per_epoch * NUM_EPOCHS:,}")

## A.7 — Training

> ⏱ Expected time: ~1.5–2 hours on T4.  
> Checkpoints are saved every epoch to Google Drive automatically.

In [ ]:
print("🚀 Starting training …")
print(f"   Model : {BASE_MODEL_ID}")
print(f"   LoRA  : r={LORA_R}, alpha={LORA_ALPHA}")
print(f"   Batch : {BATCH_SIZE} × {GRAD_ACCUM} grad accum = {BATCH_SIZE*GRAD_ACCUM} effective")
print(f"   LR    : {LEARNING_RATE} | Epochs: {NUM_EPOCHS}")
print()

train_result = trainer.train()

print("\n✅ Training complete!")
print(f"   Train loss : {train_result.training_loss:.4f}")
print(f"   Train time : {train_result.metrics['train_runtime'] / 60:.1f} min")

import shutil
shutil.copytree(LOCAL_CKPT_DIR, DRIVE_CHECKPOINT_DIR, dirs_exist_ok=True)
print(f"   Checkpoint backed up to: {DRIVE_CHECKPOINT_DIR}")

## A.8 — Evaluation on Test Set

In [ ]:
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    average_precision_score, classification_report
)

print("▶ Running inference on test set …")
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

model.eval()
all_logits, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        ids  = batch["input_ids"].to(model.device)
        mask = batch["attention_mask"].to(model.device)
        lbls = batch["labels"].numpy()
        out  = model(input_ids=ids, attention_mask=mask)
        all_logits.append(out.logits.cpu().numpy())
        all_labels.append(lbls)

import numpy as np
logits = np.vstack(all_logits)
labels = np.vstack(all_labels)
probs  = 1 / (1 + np.exp(-logits))   # sigmoid
preds  = (probs >= THRESHOLD).astype(int)

micro_f1   = f1_score(labels, preds, average="micro",   zero_division=0)
macro_f1   = f1_score(labels, preds, average="macro",   zero_division=0)
micro_prec = precision_score(labels, preds, average="micro", zero_division=0)
micro_rec  = recall_score(labels, preds, average="micro",    zero_division=0)
map_score  = average_precision_score(labels, probs, average="micro")

try:
    auc = roc_auc_score(labels, probs, average="micro")
except:
    auc = float("nan")

print("\n" + "=" * 50)
print("TEST SET RESULTS")
print("=" * 50)
print(f"  Micro-F1        : {micro_f1:.4f}")
print(f"  Macro-F1        : {macro_f1:.4f}")
print(f"  Micro-Precision : {micro_prec:.4f}")
print(f"  Micro-Recall    : {micro_rec:.4f}")
print(f"  AUC-ROC (micro) : {auc:.4f}")
print(f"  mAP (micro)     : {map_score:.4f}")
print("=" * 50)

results_dict = {
    "model": BASE_MODEL_ID, "strategy": "QLoRA",
    "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
    "max_length": MAX_LENGTH, "epochs": NUM_EPOCHS,
    "micro_f1": round(micro_f1, 4), "macro_f1": round(macro_f1, 4),
    "micro_precision": round(micro_prec, 4), "micro_recall": round(micro_rec, 4),
    "auc_roc": round(auc, 4) if not np.isnan(auc) else None,
    "mAP": round(map_score, 4),
    "top_k_codes": TOP_K_CODES, "num_train": len(train_ds),
    "threshold": THRESHOLD,
}
print("\nResults dict:", results_dict)

In [ ]:
import matplotlib.pyplot as plt

per_code_f1 = f1_score(labels, preds, average=None, zero_division=0)
code_names   = MLB.classes_

top10_idx   = np.argsort(per_code_f1)[::-1][:10]
top10_codes = [code_names[i] for i in top10_idx]
top10_f1    = [per_code_f1[i] for i in top10_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(top10_codes[::-1], top10_f1[::-1], color="steelblue")
axes[0].set_xlabel("F1 Score")
axes[0].set_title("Top-10 ICD-10 Codes by F1")
axes[0].axvline(micro_f1, color="red", linestyle="--", label=f"Micro-F1={micro_f1:.3f}")
axes[0].legend()

metric_names = ["Micro-F1", "Macro-F1", "AUC-ROC", "mAP"]
metric_vals  = [micro_f1, macro_f1, auc if not np.isnan(auc) else 0, map_score]
axes[1].bar(metric_names, metric_vals, color=["#2E6DA4", "#17A58D", "#E8A020", "#8B44D9"])
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("Score")
axes[1].set_title("Overall Evaluation Metrics (Test Set)")
for i, v in enumerate(metric_vals):
    axes[1].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(f"{LOCAL_CKPT_DIR}/evaluation_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Plot saved")

## A.9 — Save Artifacts

Saves the LoRA adapter weights (~20–50 MB) and all metadata needed for inference.

In [ ]:
import shutil, json, pickle
from pathlib import Path

ADAPTER_SAVE_DIR = "/content/icd_insight_adapters"
os.makedirs(ADAPTER_SAVE_DIR, exist_ok=True)

model.save_pretrained(ADAPTER_SAVE_DIR)
print(f"✅ LoRA adapter saved to {ADAPTER_SAVE_DIR}")

tokenizer.save_pretrained(ADAPTER_SAVE_DIR)
shutil.copy(MLB_PATH, f"{ADAPTER_SAVE_DIR}/label_binarizer.pkl")

with open(f"{ADAPTER_SAVE_DIR}/training_results.json", "w") as f:
    json.dump(results_dict, f, indent=2)

shutil.copy(f"{LOCAL_CKPT_DIR}/evaluation_results.png", f"{ADAPTER_SAVE_DIR}/evaluation_results.png")

print("\nSaved files:")
for p in sorted(Path(ADAPTER_SAVE_DIR).rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / 1e6
        print(f"  {p.name:<40} {size_mb:>8.2f} MB")

shutil.copytree(ADAPTER_SAVE_DIR, DRIVE_CHECKPOINT_DIR, dirs_exist_ok=True)
print(f"\n✅ All artifacts backed up to Drive: {DRIVE_CHECKPOINT_DIR}")

## A.10 — Download Adapters for GitHub

Run this cell to download a zip archive containing all adapter files.  
Then commit them to your GitHub repo under `checkpoints/`.

In [ ]:
import shutil
from google.colab import files

zip_path = "/content/icd_insight_adapters.zip"
shutil.make_archive("/content/icd_insight_adapters", "zip", ADAPTER_SAVE_DIR)

import os
size_mb = os.path.getsize(zip_path) / 1e6
print(f"📦 Archive size: {size_mb:.1f} MB")

if size_mb > 100:
    print("⚠ File > 100 MB — you will need Git LFS for GitHub upload")
    print("  Run: git lfs install && git lfs track 'checkpoints/*.safetensors'")
else:
    print("✅ File < 100 MB — can be committed directly to GitHub")

files.download(zip_path)

print("\n" + "=" * 60)
print("NEXT STEPS → Push adapters to GitHub")
print("=" * 60)
print("""
  1. Unzip the downloaded archive
  2. Copy files to your local icd-insight/checkpoints/ folder
  3. git add checkpoints/ results/
  4. git commit -m 'Add QLoRA adapter weights after training'
  5. git push origin main

  Then open Notebook B (02_Inference_New_Summary.ipynb) to run inference!
""")

---
## 📊 Training Summary

| Item | Value |
|------|-------|
| Base model | BioClinical ModernBERT-base |
| Strategy | QLoRA (4-bit NF4 + LoRA adapters) |
| LoRA rank | r=16, alpha=32 |
| Token length | 2048 |
| Training set | birgermoell/icd10-clinical-notes (Public HF) |
| Target codes | Top-50 ICD-10-CM |
| Loss | Focal Loss (γ=2) |
| Adapter size | ~20–50 MB |

*ICD-Insight — M.Tech Final Year Project*